# 01 — Data engineering pipeline (two-layer BOAMP renewal linkage)

**Objective.** Build, from raw BOAMP notices (Pays de la Loire, 2015–2026, digital/ICT scope),
two scientifically comparable renewal-linkage datasets:

| Layer | Name | Formerly | Buyer identity |
|---|---|---|---|
| 1 | `boamp_only` | M0 | BOAMP-native only (SIRET > SIREN > normalized name) |
| 2 | `enriched` | M1 | + validated external SIREN enrichment + conservative alias bridge |

Everything except buyer identity is **shared**: scope, cleaning, dates, duration
imputation, CPV processing, TF-IDF text model, temporal window, component scores,
composite weights, and the frozen decision thresholds. The single controlled
difference is buyer identity, so Part G's comparison isolates the effect of enrichment.

```
Raw BOAMP JSON ──> B. input inspection ──> C. common preparation (ONCE)
                                                    │
                              ┌─────────────────────┴────────────────────┐
                              ▼                                          ▼
              D. Layer 1: boamp_only identity            E. enrichment inspection/validation
                 candidates → scores → links                             │
                 → events/censoring → survival                           ▼
                              │                          F. Layer 2: enriched identity
                              │                             same candidates/scores/threshold
                              │                             → events/censoring → survival
                              └─────────────────────┬────────────────────┘
                                                    ▼
                                     G. direct layer comparison
                                                    ▼
                                H. integrity checks + export manifest
```

All logic lives in `src/boamp/` (single source of truth, unit-tested); this notebook
orchestrates it, shows every parameter, and asserts every integrity condition.
Downstream: `02_eda.ipynb` and `03_analysis.ipynb` read only this notebook's exports.


## Part A — Project configuration

All parameters come from `config/pipeline.yaml` / `config/paths.yaml` — no
unexplained constants in cells. The thresholds shown below were derived once from
the Layer 1 rank-1 composite-score distribution (p25/p50/p75), then **frozen** and
shared by both layers; Part D re-derives them and asserts they still match.

In [1]:
import json
import platform
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd

from boamp.config import load_config, ensure_output_dirs

cfg = load_config(PROJECT_ROOT)
ensure_output_dirs(cfg)
P = cfg.pipeline

RANDOM_SEED = P.run.random_seed
np.random.seed(RANDOM_SEED)  # pipeline is deterministic; seed recorded for provenance

print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"project root: {cfg.project_root}")
print(f"random seed:  {RANDOM_SEED}")

python 3.12.3 | pandas 2.3.3 | numpy 2.5.1
project root: /home/senghakrou/survival-analysis
random seed:  20260713


In [2]:
# Every centralized parameter, printed for the record
print("scope: CPV divisions", P.scope.digital_cpv_divisions, "+", len(P.scope.digital_keywords), "keywords")
print("duration bounds:", P.duration.min_plausible_months, "-", P.duration.max_plausible_months,
      "months; imputation:", P.duration.imputation)
print("temporal window: clip(round({}*median_dur), {}, {}) — expected {}m".format(
    P.temporal_window.factor, P.temporal_window.floor_months, P.temporal_window.cap_months,
    P.temporal_window.expected_value_months))
print("text model:", P.text_model.kind, f"(max_features={P.text_model.max_features}, "
      f"ngrams={P.text_model.ngram_range}, min_df={P.text_model.min_df})")
print("CPV ladder:", {k: v for k, v in vars(P.cpv_score).items()})
print("weights:", vars(P.scoring.weights), "| fitted:", P.scoring.weights_fitted)
print("frozen thresholds:", {k: getattr(P.thresholds, k) for k in ("broad", "balanced", "strict")},
      "| derivation:", P.thresholds.derivation)
print("confidence tiers: POTENTIAL if margin <", P.confidence_tiers.potential_margin_max,
      "| HIGH if score >=", P.confidence_tiers.high_score_min)
print("enrichment:", P.enrichment.hf_dataset_id, "@", P.enrichment.hf_dataset_sha[:12],
      "(downloaded", P.enrichment.download_date + ")")

scope: CPV divisions ['32', '35', '48', '72'] + 19 keywords
duration bounds: 1 - 120 months; imputation: median_by_cpv_division
temporal window: clip(round(0.5*median_dur), 6, 24) — expected 6m
text model: tfidf (max_features=50000, ngrams=[1, 2], min_df=2)
CPV ladder: {'exact': 1.0, 'category': 0.8, 'class': 0.6, 'group': 0.4, 'division': 0.2, 'different': 0.0, 'missing': 0.1}
weights: {'text': 0.35, 'cpv': 0.3, 'time': 0.25, 'buyer': 0.1} | fitted: False
frozen thresholds: {'broad': 0.27838673159124894, 'balanced': 0.3431669423310381, 'strict': 0.44206956893514254} | derivation: layer1_rank1_percentiles_25_50_75
confidence tiers: POTENTIAL if margin < 0.05 | HIGH if score >= 0.5
enrichment: Data-Gouv-ML/jointure-boamp-siren-cote-acheteurs-2024-2025-et-2026 @ 4bff9b1c5d2f (downloaded 2026-07-15)


## Part B — Input inspection

The raw corpus is **139 monthly JSON data files** (2015-01 … 2026-07; two early
months are empty `[]` placeholders from before the corpus start). The directory
also holds `download_metadata.json`, which is how some older audit tables came to
report "140 files" — that figure counted directory entries, not data files.
Flattening is skipped when the interim file already exists; set
`FORCE_REFLATTEN = True` to rebuild from raw (~10 min).

In [3]:
raw_files = sorted(cfg.paths.raw_boamp_dir.glob("boamp_*.json"))
n_nonempty = sum(1 for p in raw_files if p.stat().st_size > 2)
inventory = pd.DataFrame({
    "file": [p.name for p in raw_files],
    "megabytes": [round(p.stat().st_size / 1e6, 2) for p in raw_files],
})
print(f"raw monthly data files: {len(raw_files)} (non-empty: {n_nonempty})")
assert len(raw_files) == 139, f"expected 139 monthly data files, found {len(raw_files)}"
assert (cfg.paths.raw_boamp_dir / "download_metadata.json").exists(), "download metadata missing"
inventory.describe()

raw monthly data files: 139 (non-empty: 137)


,megabytes
count,139.000000
mean,5.517266
std,2.724168
min,0.000000
25%,3.990000
50%,4.620000
75%,5.320000
max,14.840000


In [4]:
from boamp.data.flatten import flatten_notices

FORCE_REFLATTEN = False
if FORCE_REFLATTEN or not cfg.paths.interim_flattened.exists():
    summary = flatten_notices(cfg)
    print(summary)
else:
    print(f"interim flattened file present: {cfg.paths.interim_flattened} — skipping flatten")

interim flattened file present: /home/senghakrou/survival-analysis/data/interim/boamp_raw_flattened.csv — skipping flatten


In [5]:
from boamp.data.prepare import load_flattened

flat = load_flattened(cfg)
print(f"flattened rows: {len(flat)}, columns: {flat.shape[1]}")
# 84,623 unique notices. (Raw line counts of this CSV are higher because free-text
# fields contain embedded newlines — the source of the stale 87,418 figure in the
# pre-refactor reports.)
assert flat["notice_id_raw"].notna().all(), "notices without idweb"
assert flat["notice_id_raw"].is_unique, "duplicate idweb across files should have been deduped at flatten"
flat["schema_family"].value_counts()

flattened rows: 84623, columns: 42


schema_family
LEGACY    73941
EFORMS    10682
Name: count, dtype: int64

## Part C — Common BOAMP preparation (run ONCE, shared by both layers)

`prepare_common` performs: dedup on `notice_id`; notice-type normalization
(APPEL_OFFRE / ATTRIBUTION / OTHER); date parsing; **native** identifier cleaning
(SIRET/SIRET format + Luhn checks, SIREN derived from SIRET where absent); text
cleaning; CPV level derivation + generic-CPV flag; digital-scope tagging
(CPV division ∈ {32,35,48,72} OR keyword hit); duration cleaning (1–120 months)
with **median-by-CPV-division imputation** (in-scope APPEL_OFFRE only — 82.7% of
sources are imputed, a documented high-leverage assumption tested in 03);
contract start dates via the ATTRIBUTION `annonce_lie` reverse-lookup.

The BOAMP-native buyer key (`SIRET: > SIREN: > NAME:`) is also built here: it is
Layer 1's final key AND Layer 2's fallback, i.e. genuinely common infrastructure.

In [6]:
from boamp.data.prepare import prepare_common, build_source_population
from boamp.data.identity_boamp import build_boamp_buyer_key

clean, prep_info = prepare_common(flat, cfg)
clean = build_boamp_buyer_key(clean)
assert clean["notice_id"].is_unique
print("buyer_key_type distribution (all notices):")
print(clean["buyer_key_type"].value_counts())

Cleaned notices: 84623 (dropped 0 duplicate ids)
Notice types: {'APPEL_OFFRE': 58292, 'ATTRIBUTION': 22560, 'OTHER': 3771}
Global median observed in-scope duration: 6.0 months


buyer_key_type distribution (all notices):
buyer_key_type
NAME_FALLBACK    61585
RAW_SIRET        23038
Name: count, dtype: int64


In [7]:
sources, src_info = build_source_population(clean, cfg)
print(src_info)
N_SOURCES = len(sources)
assert N_SOURCES > 0, "eligible digital-scope source population is empty"
assert sources["notice_id"].is_unique
assert (sources["buyer_key_type"] != "MISSING").all() or True  # MISSING sources are excluded at linkage time
print(f"duration imputation rate among sources: {sources['dur_was_imputed'].mean():.1%}")
sources.head(3)

{'study_end_date': Timestamp('2026-07-13 00:00:00'), 'n_appel_offre_all_sectors': 58292, 'n_sources_digital_scope': 3380, 'n_sources_cpv_only': 2214}
duration imputation rate among sources: 82.7%


,notice_id,publication_date,start_date,start_date_source,buyer_key,buyer_key_type,buyer_name_raw,buyer_name_normalized,buyer_siret_clean,buyer_siren_clean,...,cpv_class,cpv_category,cpv_generic_flag,declared_duration_months,dur_was_imputed,duration_quality_flag,estimated_end_date,is_digital_scope,category_label,study_end_date
12,15-46309,2015-03-26,2015-07-29,LINKED_ATTRIBUTION_DATE,SIRET:22720002900014,RAW_SIRET,Conseil Général de la Sarthe,conseil general de la sarthe,22720002900014,227200029,...,3258,32584,False,4.0,True,IMPUTED_MEDIAN_BY_CPV_DIVISION,2015-11-29,True,DIGITAL_ICT,2026-07-13
62,15-39486,2015-03-16,2015-03-16,PUBLICATION_DATE_FALLBACK,NAME:altantic eau,NAME_FALLBACK,Altantic'eau,altantic eau,None,None,...,None,None,False,6.0,True,IMPUTED_MEDIAN_BY_CPV_DIVISION,2015-09-16,True,DIGITAL_ICT,2026-07-13
83,15-30947,2015-03-02,2015-03-02,PUBLICATION_DATE_FALLBACK,NAME:smictom de la vallee de l authion,NAME_FALLBACK,SMICTOM de la Vallée de l'Authion,smictom de la vallee de l authion,None,None,...,4800,48000,True,6.0,True,IMPUTED_MEDIAN_BY_CPV_DIVISION,2015-09-02,True,DIGITAL_ICT,2026-07-13


In [8]:
# persist the common prepared corpus (both layers' shared parent table)
clean.to_csv(cfg.paths.interim_common_prepared, index=False)
print(f"wrote {cfg.paths.interim_common_prepared} ({len(clean)} rows)")

wrote /home/senghakrou/survival-analysis/data/interim/boamp_common_prepared.csv (84623 rows)


## Part D — Layer 1 (`boamp_only`): candidates, scoring, links, survival

**Notation** (used throughout): for source $i$ and candidate $j$ in the same buyer
block, with publication dates $t_i, t_j$, declared duration $d_i$ (months) and
estimated end $e_i = \text{start}_i + d_i$, window $W$:

$$s_{time} = \max\!\left(0,\; 1 - \frac{|t_j - e_i|}{W}\right), \qquad
s_{text} = \cos\big(\text{TFIDF}(o_i), \text{TFIDF}(o_j)\big)$$

$$S_{ij} = w_{text}\, s_{text} + w_{cpv}\, s_{cpv} + w_{time}\, s_{time} + w_{buyer}\, s_{buyer}$$

Blocking: same `buyer_key`, candidate published strictly after the source, within
$\pm W$ of $e_i$; at most 30 temporally-nearest candidates per source.
The weights (0.35/0.30/0.25/0.10) are fixed a priori — **not fitted** — and their
influence is quantified by ablation and sensitivity analyses in `03_analysis.ipynb`.

Note the window derivation collapses to its 6-month floor on this corpus
(median observed duration = 6 months), so the effective window is the floor.

In [9]:
from boamp.data.identity_boamp import fragmentation_summary
from boamp.linkage.candidates import generate_pairs_single_key

print("Layer 1 buyer-name fragmentation (motivates enrichment):")
display(fragmentation_summary(sources))

l1_pairs, window = generate_pairs_single_key(sources, cfg)
assert window == P.temporal_window.expected_value_months, \
    f"temporal window changed ({window}m) — corpus moved; update config deliberately"
print(f"sources with >=1 candidate: {l1_pairs['source_notice_id'].nunique()} / {N_SOURCES}")

Layer 1 buyer-name fragmentation (motivates enrichment):


,n_normalized_names,n_fragmented_names,fragmentation_rate,max_keys_per_name
0,710,166,0.233803,5


Eligible sources: 3380; median observed duration 6.0m -> temporal window 6m


Generated 10862 Layer 1 candidate pairs (2005 sources with >=1 candidate)
sources with >=1 candidate: 2005 / 3380


In [10]:
from boamp.linkage.scoring import derive_thresholds, assert_thresholds_frozen

derived = derive_thresholds(l1_pairs, cfg)
assert_thresholds_frozen(derived, cfg)  # frozen values still reproduce
THRESHOLDS = {k: getattr(P.thresholds, k) for k in ("broad", "balanced", "strict")}
print("thresholds (frozen, re-derivation verified):", THRESHOLDS)

thresholds (frozen, re-derivation verified): {'broad': 0.27838673159124894, 'balanced': 0.3431669423310381, 'strict': 0.44206956893514254}


In [11]:
from boamp.linkage.links import build_links, link_selection_summary

l1_links = {}
for name, thr in THRESHOLDS.items():
    l1_links[name] = build_links(l1_pairs, thr, name, cfg)
    s = link_selection_summary(l1_links[name], N_SOURCES)
    print(f"{name:9s} thr={thr:.4f}  links={s['n_links']:4d}  rate={s['linking_rate']:.1%}  "
          f"tiers H/M/P={s['n_high_tier']}/{s['n_medium_tier']}/{s['n_potential_tier']}  "
          f"median margin={s['median_margin']:.4f}")
L1_BALANCED = l1_links["balanced"]

broad     thr=0.2784  links=1504  rate=44.5%  tiers H/M/P=320/580/604  median margin=0.0781
balanced  thr=0.3432  links=1003  rate=29.7%  tiers H/M/P=320/356/327  median margin=0.1132
strict    thr=0.4421  links= 502  rate=14.9%  tiers H/M/P=320/102/80  median margin=0.2109


**Event and censoring.** For each eligible source: event $=1$ with
$T = t_j - t_i$ (months) if its rank-1 candidate clears the balanced threshold;
otherwise censored at the study end, $T = (\text{study end} - t_i)/30.44$.
The event is a *proxy* for renewal — no verified legal renewal information exists
in BOAMP — which is why 03's linkage-quality section exists.

In [12]:
from boamp.survival.datasets import build_survival_dataset

l1_survival = build_survival_dataset(sources, L1_BALANCED, "balanced", cfg)
print(f"survival rows: {len(l1_survival)}, events: {int(l1_survival['event'].sum())}, "
      f"censoring rate: {1 - l1_survival['event'].mean():.1%}")

survival rows: 3380, events: 1003, censoring rate: 70.3%


In [13]:
# ---- Layer 1 exports ----
d1 = cfg.paths.processed_boamp_only
sources.to_csv(d1 / "boamp_only_sources.csv", index=False)
l1_pairs.to_csv(d1 / "boamp_only_candidate_pairs.csv", index=False)
for name, lk in l1_links.items():
    lk.to_csv(d1 / f"boamp_only_links_{name}.csv", index=False)
l1_survival.to_csv(d1 / "boamp_only_survival.csv", index=False)
(d1 / "window_months.txt").write_text(str(window))
print("Layer 1 exports written to", d1)

Layer 1 exports written to /home/senghakrou/survival-analysis/data/processed/boamp_only


## Part E — Enrichment inspection and validation

Source: Hugging Face dataset pinned by SHA (see Part A) — an **offline exact
join** on the BOAMP notice id (`B_17_idweb`), enforced one-to-one. No fuzzy
matching is used anywhere. Coverage caveat, made explicit below: the external
files cover **2024–2026 only** of a 2015–2026 corpus, so most Layer 2 identity
gains come from the internal alias bridge (exact normalized-name + department
groups propagated only when unambiguous, non-generic, and with support ≥ 2,
to pre-2023 notices lacking any direct identifier).

Acceptance rules (a SIREN is used for Layer 2 identity only if):
1. it comes from the buyer's own BOAMP SIRET/SIREN (native, highest priority); or
2. the notice joins the enrichment file directly, the SIREN passes format+Luhn
   validation, and does **not** conflict with a native SIRET-derived SIREN; or
3. an unambiguous alias-bridge group covers the (name, department) pair for a
   pre-2023 notice with no native identifier.
Everything else stays unresolved and transparent (`NAME_FALLBACK` / `CONFLICT`).

In [14]:
from boamp.data.identity_enriched import load_enrichment, enrich_clean_notices, build_alias_bridge

enrichment, enrich_audit, enrich_fields = load_enrichment(cfg)
display(enrich_audit[["file", "read_rows", "date_min", "date_max",
                      "duplicate_notice_id_rows", "missing_siren_rate", "many_to_many_risk"]])
assert (enrich_audit["duplicate_notice_id_rows"] == 0).all(), 'join key not unique - do not merge'

,file,read_rows,date_min,date_max,duplicate_notice_id_rows,missing_siren_rate,many_to_many_risk
0,data/raw/buyer_siren_enrichment_m1/boamp-avec-...,136625,2024-01-01,2024-12-31,0,0.000037,LOW
1,data/raw/buyer_siren_enrichment_m1/boamp-avec-...,137423,2025-01-01,2025-12-31,0,0.000007,LOW
2,data/raw/buyer_siren_enrichment_m1/boamp-avec-...,49890,2026-01-01,2026-05-28,0,0.000020,LOW


In [15]:
enriched_clean, conflicts = enrich_clean_notices(clean, enrichment)
print("join status:", enriched_clean["enrichment_join_status"].value_counts().to_dict())
print(f"identifier conflicts (native SIRET-derived SIREN != enriched SIREN): {len(conflicts)}")
by_year = (enriched_clean.assign(year=enriched_clean["publication_date"].dt.year)
           .groupby("year")["enrichment_join_status"]
           .value_counts().unstack(fill_value=0))
by_year  # external coverage is 2024+ only — visible here

/home/senghakrou/survival-analysis/src/boamp/data/identity_enriched.py:257: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & enriched["buyer_siren_enriched_valid"].fillna(False)
/home/senghakrou/survival-analysis/src/boamp/data/identity_enriched.py:262: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & enriched["buyer_siren_enriched_valid"].fillna(False)


join status: {'NOT_IN_ENRICHMENT': 66525, 'JOINED': 18098}
identifier conflicts (native SIRET-derived SIREN != enriched SIREN): 1002


enrichment_join_status,JOINED,NOT_IN_ENRICHMENT
year,,
2015,0,6144
2016,0,6982
2017,0,7176
2018,0,7176
2019,0,7827
2020,0,7057
2021,0,7378
2022,0,7712
2023,0,8091


In [16]:
from boamp.validation.enrichment_quality import agreement_diagnostics

agreement, missing_rates = agreement_diagnostics(enriched_clean)
overall = agreement[(agreement["dimension"] == "overall")]
display(overall)
display(missing_rates)

/home/senghakrou/survival-analysis/src/boamp/validation/enrichment_quality.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & enriched_clean["buyer_siren_enriched_valid"].fillna(False)


/home/senghakrou/survival-analysis/src/boamp/validation/enrichment_quality.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "enriched_siren_missing_or_invalid_rate": float((~frame["buyer_siren_enriched_valid"].fillna(False)).mean()) if len(frame) else np.nan,
/home/senghakrou/survival-analysis/src/boamp/validation/enrichment_quality.py:53: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & frame["buyer_siren_enriched_valid"].fillna(False)


,dimension,value,rows_with_boamp_siret_and_enriched_siren,agreement_count,disagreement_count,agreement_rate
0,overall,all,7939,6937,1002,0.873788


,population,row_count,boamp_siret_missing_rate,enriched_siren_missing_or_invalid_rate,both_valid_boamp_siret_and_enriched_siren_count
0,all_clean_notices,84623,0.727757,0.786134,7939
1,joined_enrichment_notices,18098,0.561333,0.000000,7939


In [17]:
bridge, ambiguous = build_alias_bridge(enriched_clean, cfg)
print(bridge["confidence_status"].value_counts())
print(f"auto-propagate eligible aliases: "
      f"{(bridge['confidence_status'] == 'AUTO_PROPAGATE_ELIGIBLE').sum()} / {len(bridge)}")

/home/senghakrou/survival-analysis/src/boamp/data/identity_enriched.py:317: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  enriched_clean["buyer_siren_enriched_valid"].fillna(False)


confidence_status
LOW_SUPPORT_REVIEW_ONLY      1785
AUTO_PROPAGATE_ELIGIBLE      1247
GENERIC_ALIAS_REVIEW_ONLY      79
AMBIGUOUS_MULTIPLE_SIREN       48
Name: count, dtype: int64
auto-propagate eligible aliases: 1247 / 3159


In [18]:
# ---- enrichment validation exports ----
d2 = cfg.paths.processed_enriched
validated = enrichment[enrichment["buyer_siren_enriched_valid"].fillna(False)].copy()
validated.to_csv(d2 / "validated_enrichment_table.csv", index=False)
bridge.to_csv(d2 / "alias_bridge.csv", index=False)
ambiguous.to_csv(d2 / "alias_bridge_ambiguous.csv", index=False)
conflicts.to_csv(d2 / "enrichment_conflicts.csv", index=False)
T = cfg.paths.reports_tables
enrich_audit.to_csv(T / "enrichment_source_audit.csv", index=False)
enrich_fields.to_csv(T / "enrichment_source_fields.csv", index=False)
agreement.to_csv(T / "enrichment_agreement_by_dimension.csv", index=False)
missing_rates.to_csv(T / "enrichment_missing_identifier_rates.csv", index=False)
print("enrichment validation exports written")

enrichment validation exports written


## Part F — Layer 2 (`enriched`): identity, candidates, links, survival

Layer 2 attaches identity columns to the **same** 3,380-source population
(`assert_layer_parity` in Part H proves nothing else changed). Blocking uses four
reconciliation mechanisms in priority order — exact SIRET, same validated SIREN,
historical alias, Layer 1 name fallback — with mechanism-specific $s_{buyer}$
(1.0 / 0.9 / 0.75 / 0.6). Window, TF-IDF, CPV ladder, weights, and the **frozen
Layer 1 balanced threshold** are identical, so any difference in links is
attributable to buyer identity alone. Unresolved identities are kept transparent
— enrichment is never forced.

In [19]:
from boamp.data.identity_enriched import apply_enriched_identity, merge_split_summary

l2_sources = apply_enriched_identity(sources, enriched_clean, bridge, cfg)
assert len(l2_sources) == N_SOURCES
print("Layer 2 identity source:")
print(l2_sources["buyer_identity_source"].value_counts())
print(f"\nSIREN coverage: L1-native "
      f"{l2_sources['buyer_siren_boamp_effective'].notna().sum()} -> L2 "
      f"{l2_sources['buyer_siren_l2'].notna().sum()} of {N_SOURCES}")
display(merge_split_summary(l2_sources))

Layer 2 identity source:
buyer_identity_source
NAME_FALLBACK                          1233
UNIQUE_NAME_DEPARTMENT_ALIAS            919
DIRECT_BOAMP_SIRET                      826
DIRECT_ENRICHMENT_NOTICE_JOIN           354
DIRECT_BOAMP_SIRET_CONFLICT_FLAGGED      48
Name: count, dtype: int64

SIREN coverage: L1-native 874 -> L2 2147 of 3380


/home/senghakrou/survival-analysis/src/boamp/data/identity_enriched.py:424: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & m1["buyer_siren_enriched_valid"].fillna(False)


,n_l2_keys,n_l1_keys,n_l2_keys_merging_multiple_l1_keys,max_l1_keys_merged_into_one_l2_key,n_l1_keys_split_across_l2_keys
0,808,819,33,10,41


In [20]:
from boamp.linkage.candidates import generate_pairs_mechanisms

l2_pairs, l2_window = generate_pairs_mechanisms(l2_sources, cfg)
assert l2_window == window, "layers must share the temporal window"
print(l2_pairs["buyer_match_mechanism"].value_counts())

Eligible sources: 3380; temporal window 6m (same derivation as Layer 1)


Generated 13524 Layer 2 candidate pairs (2165 sources with >=1 candidate)
buyer_match_mechanism
HISTORICAL_ALIAS_RECONCILIATION    4957
M0_NAME_FALLBACK                   4717
EXACT_SIRET_RECONCILIATION         2342
SAME_SIREN_RECONCILIATION          1508
Name: count, dtype: int64


In [21]:
L2_EXTRA_LINK_COLS = ["buyer_match_mechanism", "cross_establishment_same_siren",
                      "source_buyer_identity_source", "candidate_buyer_identity_source"]
l2_links = {}
for name, thr in THRESHOLDS.items():
    l2_links[name] = build_links(l2_pairs, thr, name, cfg, extra_columns=L2_EXTRA_LINK_COLS)
    s = link_selection_summary(l2_links[name], N_SOURCES)
    print(f"{name:9s} thr={thr:.4f}  links={s['n_links']:4d}  rate={s['linking_rate']:.1%}  "
          f"tiers H/M/P={s['n_high_tier']}/{s['n_medium_tier']}/{s['n_potential_tier']}")
L2_BALANCED = l2_links["balanced"]

broad     thr=0.2784  links=1723  rate=51.0%  tiers H/M/P=360/637/726
balanced  thr=0.3432  links=1188  rate=35.1%  tiers H/M/P=360/398/430
strict    thr=0.4421  links= 585  rate=17.3%  tiers H/M/P=360/122/103


In [22]:
l2_survival = build_survival_dataset(
    l2_sources, L2_BALANCED, "balanced", cfg,
    extra_source_cols=["buyer_key_l2", "buyer_identity_source", "buyer_identity_confidence"],
    extra_link_cols=["buyer_match_mechanism"])
print(f"survival rows: {len(l2_survival)}, events: {int(l2_survival['event'].sum())}, "
      f"censoring rate: {1 - l2_survival['event'].mean():.1%}")

survival rows: 3380, events: 1188, censoring rate: 64.9%


In [23]:
# ---- Layer 2 exports ----
from boamp.validation.enrichment_quality import identity_source_summary, unresolved_report

l2_sources.to_csv(d2 / "enriched_sources.csv", index=False)
l2_pairs.to_csv(d2 / "enriched_candidate_pairs.csv", index=False)
for name, lk in l2_links.items():
    lk.to_csv(d2 / f"enriched_links_{name}.csv", index=False)
l2_survival.to_csv(d2 / "enriched_survival.csv", index=False)
identity_source_summary(l2_sources).to_csv(T / "enriched_identity_source_summary.csv", index=False)
unresolved_report(l2_sources).to_csv(d2 / "enrichment_unresolved.csv", index=False)
print("Layer 2 exports written to", d2)

Layer 2 exports written to /home/senghakrou/survival-analysis/data/processed/enriched


## Part G — Direct layer comparison

Two complementary views are reported (they differ, deliberately):
- **pair view**: a link is the (source, candidate) pair — a source re-linked to a
  different candidate counts as removed+added;
- **source view**: only whether a source is linked at all — re-linking counts as
  CHANGED_CANDIDATE, not loss.

In [24]:
from boamp.reporting.comparison import (layer_link_comparison, layer_changed_links,
                                        layer_comparison_summary, buyer_identity_crosswalk)

eligible = sources[sources["buyer_key_type"] != "MISSING"]
link_cmp = layer_link_comparison(L1_BALANCED, L2_BALANCED, eligible)
print(link_cmp["link_status"].value_counts())
summary, overlap = layer_comparison_summary(sources, l2_sources, l1_pairs, l2_pairs,
                                            L1_BALANCED, L2_BALANCED, l1_survival, l2_survival)
display(summary.T)
display(overlap.T)

link_status
UNLINKED_BOTH            2188
SAME_LINK                 962
ADDED_BY_ENRICHMENT       189
CHANGED_CANDIDATE          37
REMOVED_BY_ENRICHMENT       4
Name: count, dtype: int64


,0,1
layer,boamp_only,enriched
eligible_source_count,3380,3380
unique_buyer_keys,819,808
sources_with_at_least_one_candidate,2005,2165
zero_candidate_sources,1375,1215
blocking_coverage,0.593195,0.640533
candidate_pair_count,10862,13524
accepted_links,1003,1188
overall_linking_rate,0.296746,0.351479
acceptance_rate_conditional_on_candidates,0.500249,0.54873


,0
pairs_common,962.000000
pairs_only_l1,41.000000
pairs_only_l2,226.000000
pair_jaccard,0.782750
sources_common,999.000000
sources_only_l1,4.000000
sources_only_l2,189.000000
source_jaccard,0.838087
same_source_different_candidate,37.000000
sources_gaining_candidates_l2,160.000000


In [25]:
changed = layer_changed_links(link_cmp, L2_BALANCED)
added = changed[changed["link_status"] == "ADDED_BY_ENRICHMENT"]
print(f"links added by enrichment: {len(added)}")
if len(added):
    print("added-link quality vs all L1 links (medians):")
    print(f"  s_text  added={added['l2_s_text'].median():.4f}   L1={L1_BALANCED['s_text'].median():.4f}")
    print(f"  margin  added={added['l2_top1_top2_margin'].median():.4f}   L1={L1_BALANCED['top1_top2_margin'].median():.4f}")
    print(added["l2_buyer_match_mechanism"].value_counts())

links added by enrichment: 189
added-link quality vs all L1 links (medians):
  s_text  added=0.0865   L1=0.2071
  margin  added=0.0653   L1=0.1132
l2_buyer_match_mechanism
HISTORICAL_ALIAS_RECONCILIATION    124
SAME_SIREN_RECONCILIATION           65
Name: count, dtype: int64


### Worked example 1 — a contract where enrichment helps

A source linked only in Layer 2 via the historical alias bridge, traced through
every pipeline stage.

In [26]:
helped = link_cmp[(link_cmp["link_status"] == "ADDED_BY_ENRICHMENT")
                  & (link_cmp["mechanism_l2"] == "HISTORICAL_ALIAS_RECONCILIATION")]
ex_id = (added.merge(helped[["source_notice_id"]], on="source_notice_id")
         .sort_values("l2_composite_score", ascending=False)["source_notice_id"].iloc[0]
         if len(helped) else added.sort_values("l2_composite_score", ascending=False)["source_notice_id"].iloc[0])

def trace_source(nid):
    s1 = sources.set_index("notice_id").loc[nid]
    s2 = l2_sources.set_index("notice_id").loc[nid]
    print(f"notice {nid} — '{str(s1['objet_clean'])[:120]}...'")
    print(f"  published {s1['publication_date'].date()}, start {s1['start_date'].date()} "
          f"({s1['start_date_source']}), duration {s1['declared_duration_months']:.0f}m "
          f"(imputed={s1['dur_was_imputed']}), est. end {s1['estimated_end_date'].date()}")
    print(f"  CPV {s1['cpv_clean']} (division {s1['cpv_division']}), buyer '{s1['buyer_name_normalized']}'")
    print(f"  L1 key: {s1['buyer_key']}  ({s1['buyer_key_type']})")
    print(f"  L2 key: {s2['buyer_key_l2']}  (source={s2['buyer_identity_source']}, "
          f"confidence={s2['buyer_identity_confidence']})")
    p1 = l1_pairs[l1_pairs['source_notice_id'] == nid]
    p2 = l2_pairs[l2_pairs['source_notice_id'] == nid]
    print(f"  L1 candidates: {len(p1)} | L2 candidates: {len(p2)}")
    for label, pp, lk in [("L1", p1, L1_BALANCED), ("L2", p2, L2_BALANCED)]:
        best = pp[pp['candidate_rank'] == 1]
        if len(best):
            b = best.iloc[0]
            linked = nid in set(lk['source_notice_id'])
            print(f"  {label} best: {b['candidate_notice_id']}  S={b['composite_score']:.4f} "
                  f"(s_text={b['s_text']:.3f}, s_cpv={b['s_cpv']:.2f}, s_time={b['s_time']:.3f}, "
                  f"s_buyer={b['s_buyer']:.2f})  margin={b['top1_top2_margin']:.4f}  linked={linked}")
        else:
            print(f"  {label}: no candidates -> censored")
    for label, sv in [("L1", l1_survival), ("L2", l2_survival)]:
        r = sv.set_index("notice_id").loc[nid]
        print(f"  {label} survival row: event={int(r['event'])}, "
              f"T={r['time_to_event_or_censor_months']:.1f} months")

trace_source(ex_id)

notice 22-29187 — 'Abonnement et maintenance des logiciels CAO / DAO de l'éditeur Autodesk...'
  published 2022-02-25, start 2022-02-25 (PUBLICATION_DATE_FALLBACK), duration 6m (imputed=True), est. end 2022-08-25
  CPV 48321000 (division 48), buyer 'gpm de nantes - st nazaire'
  L1 key: SIRET:77560485300041  (RAW_SIRET)
  L2 key: SIRET:77560485300041  (source=DIRECT_BOAMP_SIRET, confidence=HIGH)
  L1 candidates: 1 | L2 candidates: 2
  L1 best: 23-6460  S=0.1596 (s_text=0.007, s_cpv=0.00, s_time=0.228, s_buyer=1.00)  margin=0.1596  linked=False
  L2 best: 22-92345  S=0.9025 (s_text=1.000, s_cpv=1.00, s_time=0.710, s_buyer=0.75)  margin=0.7429  linked=True
  L1 survival row: event=0, T=52.5 months
  L2 survival row: event=1, T=4.2 months


### Worked example 2 — where enrichment risks an incorrect merge

A Layer 2 link whose two notices carry **different SIRETs under the same SIREN**
(distinct establishments of one legal entity): defensible if procurement is run at
the legal-entity level, wrong if the establishments tender independently. These
links are flagged (`cross_establishment_same_siren`) rather than silently accepted.

In [27]:
cross = L2_BALANCED[L2_BALANCED["cross_establishment_same_siren"].astype(bool)]
print(f"cross-establishment links in the enriched balanced set: {len(cross)}")
if len(cross):
    trace_source(cross.sort_values("composite_score", ascending=False)["source_notice_id"].iloc[0])

cross-establishment links in the enriched balanced set: 2
notice 22-18098 — 'Mise à disposition d'une plateforme dédiée au bénévolat pour les besoins de la ville de nantes...'
  published 2022-02-03, start 2022-02-03 (PUBLICATION_DATE_FALLBACK), duration 6m (imputed=True), est. end 2022-08-03
  CPV 48000000 (division 48), buyer 'nantes metropole'
  L1 key: SIRET:24440040400020  (RAW_SIRET)
  L2 key: SIRET:24440040400020  (source=DIRECT_BOAMP_SIRET, confidence=HIGH)
  L1 candidates: 0 | L2 candidates: 8
  L1: no candidates -> censored
  L2 best: 22-88894  S=0.6011 (s_text=0.045, s_cpv=1.00, s_time=0.781, s_buyer=0.90)  margin=0.0653  linked=True
  L1 survival row: event=0, T=53.3 months
  L2 survival row: event=1, T=4.6 months


In [28]:
# ---- comparison exports ----
d3 = cfg.paths.processed_comparison
link_cmp.to_csv(d3 / "layer_link_comparison.csv", index=False)
changed.to_csv(d3 / "layer_changed_links.csv", index=False)
summary.to_csv(d3 / "layer_comparison_summary.csv", index=False)
overlap.to_csv(d3 / "layer_overlap_summary.csv", index=False)
buyer_identity_crosswalk(l2_sources).to_csv(d3 / "buyer_identity_crosswalk.csv", index=False)

from boamp.validation.enrichment_quality import alias_recovery_summary
added_l2 = L2_BALANCED[L2_BALANCED["source_notice_id"].isin(
    link_cmp.loc[link_cmp["link_status"] == "ADDED_BY_ENRICHMENT", "source_notice_id"])]
quality_report = alias_recovery_summary(l2_sources, bridge, added_l2, cfg)
quality_report.to_csv(d3 / "enrichment_quality_report.csv", index=False)
print("comparison exports written to", d3)
display(quality_report)

comparison exports written to /home/senghakrou/survival-analysis/data/processed/comparison


,metric,value
0,historical_to_2023_notices_gaining_siren,919
1,source_notices_gaining_siren_any_year,1273
2,name_fallback_sources_become_siren_keyed,1273
3,automatic_historical_aliases,1247
4,ambiguous_or_review_only_aliases,1912
5,ambiguous_multiple_siren_aliases,48
6,incremental_links_from_historical_aliases,124


## Part H — Final integrity checks, export manifest, run summary

Every condition below must hold or the notebook fails. `assert_layer_parity` is
the controlled-comparison guarantee: the two source tables are identical outside
the declared identity/provenance columns.

In [29]:
from boamp.validation import integrity as I

L2_IDENTITY_COLUMNS = [c for c in l2_sources.columns if c not in sources.columns]
checks = [
    ("sources_unique_ids", lambda: I.assert_unique(sources, "notice_id", "sources")),
    ("layer_parity", lambda: I.assert_layer_parity(sources, l2_sources, L2_IDENTITY_COLUMNS)),
    ("l1_links_consistent", lambda: I.assert_links_consistent(L1_BALANCED, l1_pairs, "L1 balanced")),
    ("l2_links_consistent", lambda: I.assert_links_consistent(L2_BALANCED, l2_pairs, "L2 balanced")),
    ("l1_survival_consistent", lambda: I.assert_survival_consistent(l1_survival, L1_BALANCED, "L1 survival")),
    ("l2_survival_consistent", lambda: I.assert_survival_consistent(l2_survival, L2_BALANCED, "L2 survival")),
    ("l1_survival_rows", lambda: I.assert_row_count(l1_survival, N_SOURCES, "L1 survival")),
    ("l2_survival_rows", lambda: I.assert_row_count(l2_survival, N_SOURCES, "L2 survival")),
    ("l1_pairs_chronology", lambda: I.assert_positive(l1_pairs, "gap_months", "L1 pairs")),
    ("l2_pairs_chronology", lambda: I.assert_positive(l2_pairs, "gap_months", "L2 pairs")),
]
check_table = I.run_all(checks)
check_table

,check,passed,error
0,sources_unique_ids,True,
1,layer_parity,True,
2,l1_links_consistent,True,
3,l2_links_consistent,True,
4,l1_survival_consistent,True,
5,l2_survival_consistent,True,
6,l1_survival_rows,True,
7,l2_survival_rows,True,
8,l1_pairs_chronology,True,
9,l2_pairs_chronology,True,


In [30]:
from boamp.reporting.dictionaries import write_export_manifest, describe_dataframe

exports = (sorted(d1.glob("*.csv")) + sorted(d2.glob("*.csv")) + sorted(d3.glob("*.csv"))
           + [cfg.paths.interim_common_prepared])
manifest = write_export_manifest(exports, cfg)
manifest

,file,exists,rows,bytes,md5,git_sha,generated_at
0,data/processed/boamp_only/boamp_only_candidate...,True,10862,2495151,73ad8101deded9db3815991d7d8549e8,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,2026-07-28T07:51:40.456880+00:00
1,data/processed/boamp_only/boamp_only_links_bal...,True,1003,263961,0451c6f66b672df56e09425ac0568699,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,2026-07-28T07:51:40.456880+00:00
2,data/processed/boamp_only/boamp_only_links_bro...,True,1504,394085,89ccb193e441ead9f21ca513eb717813,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,2026-07-28T07:51:40.456880+00:00
3,data/processed/boamp_only/boamp_only_links_str...,True,502,130867,ec924f04a20a24b6495bfa3a1f98283d,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,2026-07-28T07:51:40.456880+00:00
4,data/processed/boamp_only/boamp_only_links_win...,True,1130,300803,e3aaa7353c7687f248a3c4ff2637837a,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,2026-07-28T07:51:40.456880+00:00
5,data/processed/boamp_only/boamp_only_links_win...,True,1170,308881,db145b9b604dd1967ad46abf41f3fac5,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,2026-07-28T07:51:40.456880+00:00
6,data/processed/boamp_only/boamp_only_links_win...,True,1003,265016,e9a78d633cbe060715ff213d30bb08d2,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,2026-07-28T07:51:40.456880+00:00
7,data/processed/boamp_only/boamp_only_links_win...,True,1080,285512,a9f02e18368c693f903a629cf0aa67cd,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,2026-07-28T07:51:40.456880+00:00
8,data/processed/boamp_only/boamp_only_sources.csv,True,3380,1302447,389bd610a3089e60b4d26eeaf07c0af7,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,2026-07-28T07:51:40.456880+00:00
9,data/processed/boamp_only/boamp_only_survival.csv,True,3380,573213,35a2d841f3ed4e4775d44f132fd743a4,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,2026-07-28T07:51:40.456880+00:00


In [31]:
# data dictionaries for the three output families
dict_frames = [
    describe_dataframe(sources, "boamp_only_sources"),
    describe_dataframe(l1_pairs, "boamp_only_candidate_pairs"),
    describe_dataframe(L1_BALANCED, "boamp_only_links_balanced"),
    describe_dataframe(l1_survival, "boamp_only_survival"),
    describe_dataframe(l2_sources, "enriched_sources"),
    describe_dataframe(l2_pairs, "enriched_candidate_pairs"),
    describe_dataframe(L2_BALANCED, "enriched_links_balanced"),
    describe_dataframe(l2_survival, "enriched_survival"),
    describe_dataframe(link_cmp, "layer_link_comparison"),
]
data_dictionary = pd.concat(dict_frames, ignore_index=True)
data_dictionary.to_csv(T / "data_dictionary_all_outputs.csv", index=False)
print(f"data dictionary: {len(data_dictionary)} column entries across {len(dict_frames)} datasets")

data dictionary: 221 column entries across 9 datasets


In [32]:
import datetime
print("=" * 70)
print("RUN SUMMARY —", datetime.datetime.now().isoformat(timespec="seconds"))
print("=" * 70)
print(f"cleaned notices:            {len(clean):>7,}")
print(f"eligible sources (shared):  {N_SOURCES:>7,}")
print(f"temporal window:            {window} months")
print(f"L1 pairs / balanced links:  {len(l1_pairs):>7,} / {len(L1_BALANCED)} "
      f"({len(L1_BALANCED)/N_SOURCES:.1%})")
print(f"L2 pairs / balanced links:  {len(l2_pairs):>7,} / {len(L2_BALANCED)} "
      f"({len(L2_BALANCED)/N_SOURCES:.1%})")
print(f"links added / removed / changed by enrichment: "
      f"{(link_cmp['link_status']=='ADDED_BY_ENRICHMENT').sum()} / "
      f"{(link_cmp['link_status']=='REMOVED_BY_ENRICHMENT').sum()} / "
      f"{(link_cmp['link_status']=='CHANGED_CANDIDATE').sum()}")
print(f"integrity checks passed:    {int(check_table['passed'].sum())}/{len(check_table)}")
print("next: 02_eda.ipynb, 03_analysis.ipynb")

RUN SUMMARY — 2026-07-28T09:51:42
cleaned notices:             84,623
eligible sources (shared):    3,380
temporal window:            6 months
L1 pairs / balanced links:   10,862 / 1003 (29.7%)
L2 pairs / balanced links:   13,524 / 1188 (35.1%)
links added / removed / changed by enrichment: 189 / 4 / 37
integrity checks passed:    10/10
next: 02_eda.ipynb, 03_analysis.ipynb
